# 06 Evaluate

Scores each arm with the serving engine: on dev while tuning τ, then once per (arm, seed) on the frozen test set.
Writes `evals/results/{arm}_s{seed}_{split}.json` and `..._verdicts.jsonl`, general capability per arm
(`general_capability.json`, AuditKIT on Hindi MMMLU), and the summary with regression gates
(`summary_test.{json,md}`, `headline_test.png`). Manifest: `evals/results/test_usage.json` records when each
(arm, seed) used the test set; `run_eval` refuses a second test run without `--force`.

**Needs:** a GPU (L4) and `hf auth login`; `JUDGE_MODEL` enables the LLM grader (else `--no-judge` scoring).

In [ ]:
import json, os, sys
from collections import Counter
from pathlib import Path

# Run from the repo root so relative paths match the README runbook.
ROOT = Path.cwd()
while not (ROOT / "core").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
PY = sys.executable  # `!{PY} -m ...` runs the CLI with this kernel's interpreter
print("repo root:", ROOT)

In [ ]:
def adapter(arm, seed):
    """Local adapter path for (arm, seed); edit to use HF repo ids instead."""
    if arm == "A0":
        return None
    sub = "final" if arm == "A3" else ""
    found = sorted(Path("checkpoints").glob(f"{arm}-s{seed}-*"))
    return str(found[-1] / sub) if found else None


RUNS = [("A0", 0)] + [(arm, s) for arm in ("A1", "A2") for s in (0, 1, 2)] + [("A3", 0)]

In [ ]:
!{PY} -m evals.run_eval --arm A2 --seed 0 --split dev

**Stop here.** Put adapter paths in `configs/serving.yaml` and tune `tau` in `configs/retrieval.yaml` on dev
so over-refusal stays under 10%. The next cell uses the test set, once per (arm, seed).

In [ ]:
for arm, seed in RUNS:
    path = adapter(arm, seed)
    adapter_arg = f"--adapter {path}" if path else ""
    !{PY} -m evals.run_eval --arm {arm} --seed {seed} --split test {adapter_arg}

In [ ]:
!{PY} -m adapters.auditkit_eval --arms A0 A1 A2 A3

In [ ]:
!{PY} -m evals.summarize --split test

## Summary

In [ ]:
results = Path("evals/results")
for f in sorted(results.glob("A*_s*_test.json")):
    r = json.loads(f.read_text())
    print(f.stem, {k: v["value"] for k, v in r["metrics"].items()})
md_path = results / "summary_test.md"
print(md_path.read_text(encoding="utf-8") if md_path.exists() else "(no summary yet)")